# Module 3 — FastAPI + Testing

**Deep-practice notebook:** typed contracts, streaming, cancellation, dependency boundaries, failure injection, security and measurable API reliability.

Predict → Build → Break → Debug → Measure → Defend.


## Objectives + concept map
Concept map: HTTP contract → validation → dependency/provider boundary → timeout/cancellation → error envelope → telemetry → tests → release gate.

Core principle: the API contract should remain deterministic even when the model behind it is probabilistic.

In [ ]:
!pip -q install fastapi httpx pydantic
from fastapi import FastAPI
from pydantic import BaseModel, Field
import httpx, asyncio, time
app=FastAPI()
class Ask(BaseModel): prompt:str=Field(min_length=1,max_length=500)
@app.post('/ask')
async def ask(x:Ask): return {'answer':'demo: '+x.prompt,'request_id':'demo-1'}


## BUILD — contract test
Test valid input first. Then deliberately violate the contract and inspect status code and error shape. Do not accept 'it threw an exception' as a sufficient API contract.

In [ ]:
async def test_api():
    transport=httpx.ASGITransport(app=app)
    async with httpx.AsyncClient(transport=transport,base_url='http://test') as c:
        for payload in [{'prompt':'hello'},{'prompt':''},{'prompt':'x'*501},{}]:
            r=await c.post('/ask',json=payload); print(payload if len(str(payload))<30 else 'oversized', '=>', r.status_code, r.json())
asyncio.run(test_api())


## TRY — deterministic error envelope
TODO: define `{request_id, error_code, message, retryable}`. Map provider timeout to a retryable upstream error, validation to a non-retryable client error, and cancellation to a cancelled outcome.


## LAB — streaming + cancellation
Implement a token generator that yields partial output. Cancel the client halfway through. Add cleanup instrumentation and prove that cancellation does not leave a background task running.


## BREAK — failure injection
Inject provider timeout, malformed provider response, client disconnect, slow client/backpressure, dependency failure and rate-limit responses. For each failure write: expected HTTP outcome, retry boundary, telemetry event, and test that prevents regression.

In [ ]:
failure_matrix = {
 'validation':'4xx / no retry / validation event',
 'provider_timeout':'5xx or mapped upstream / retry below API / timeout event',
 'client_cancel':'cancelled / cleanup / cancellation event',
 'rate_limit':'429 / bounded backoff below API / rate-limit event',
}
for k,v in failure_matrix.items(): print(k,'=>',v)


## Industry scenario — university AI platform
A campus service may receive thousands of simultaneous student requests. Design request limits, authentication, provider timeout, concurrency control, graceful shutdown and p95 latency SLO. Explain what the user sees during provider degradation.


## MEASURE
Capture p50/p95/p99 latency, throughput, error rate, cancellation rate, retry count, provider timeout rate and cost/request. Compare before/after bounded concurrency. Never optimize average latency while p95 and error rate deteriorate.


## Security lab
Add an auth dependency and tenant identifier. Prove one tenant cannot access another tenant's request state. Add a security regression test for missing/invalid credentials.


## Reference solution
Validate inputs at the boundary; return stable typed errors with request IDs; isolate provider retries below the HTTP contract; propagate cancellation; bound concurrency; instrument every request; expose readiness separately from liveness; enforce auth/tenant checks outside the model; and test failure behavior with deterministic fakes.


## Extension challenges
1. Add dependency overrides. 2. Add rate limiting. 3. Add idempotency keys. 4. Add streaming cleanup tests. 5. Add property tests for input bounds. 6. Add graceful shutdown. 7. Add auth/tenant isolation. 8. Add p95 latency regression gate. 9. Add provider circuit breaker. 10. Add structured trace propagation.

### Mastery gate
You can prove the API's success, failure, cancellation, security and latency behavior with executable tests and explain exactly where retries belong.